**Importación de librerías**

In [1]:
# Importamos TensorFlow
import tensorflow as tf

**Importación de Dataset**

In [2]:
# descargamos el dataset de kaggle
!pip install -q kagglehub

import kagglehub
import os

# guardamos la ruta donde se descargaron las imagenes
ruta_dataset = kagglehub.dataset_download(
    "amandam1/healthy-vs-diseased-leaf-image-dataset"
)

print("dataset descargado en:", ruta_dataset)

100%|██████████| 6.20G/6.20G [01:11<00:00, 93.1MB/s]

Extracting files...


dataset descargado en: /root/.cache/kagglehub/datasets/amandam1/healthy-vs-diseased-leaf-image-dataset/versions/1


**Selección de datos**

In [3]:
# definimos la ruta donde estan las imagenes de limon
ruta_limon = os.path.join(
    ruta_dataset,
    "A Database of Leaf Images",
    "Lemon (P10)"
)

# cargamos las imagenes para entrenar el modelo
train_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_limon,          # carpeta donde estan las imagenes
    validation_split=0.2, # usamos el 20% para validar y el 80% para entrenar
    subset="training",    # seleccionamos las imagenes de entrenamiento
    seed=42,              # mantenemos la misma division en cada ejecucion
    image_size=(224, 224),# cambiamos el tamaño de las imagenes a 224 x 224
    batch_size=16         # procesamos 16 imagenes por grupo
)

# cargamos las imagenes que usaremos para validar los resultados
val_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_limon,           # usamos la misma carpeta de imagenes
    validation_split=0.2, # reservamos el mismo 20% para validacion
    subset="validation",  # seleccionamos las imagenes de validacion
    seed=42,              # usamos la misma semilla para no cambiar la division
    image_size=(224, 224),# todas las imagenes tendran el mismo tamaño
    batch_size=16         # procesamos 16 imagenes por grupo
)

# mostramos los nombres de las clases que encontro tensorflow
print("clases:", train_ds.class_names)

Found 236 files belonging to 2 classes.
Using 189 files for training.
Found 236 files belonging to 2 classes.
Using 47 files for validation.
clases: ['diseased', 'healthy']


**Hugging Face**

In [4]:
# importamos las herramientas de hugging face
!pip install -q huggingface_hub transformers

# importamos la funcion para iniciar sesion
from huggingface_hub import notebook_login
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

# iniciamos sesion con nuestro token
notebook_login()

# revisamos que la cuenta de hugging face este conectada
api = HfApi()
usuario = api.whoami()
print("cuenta conectada:", usuario["name"])

# descargamos los modelos desde hugging face usando la sesion iniciada
rutas_modelos = {}

# guardamos los repositorios de hugging face que vamos a descargar
repositorios = {
    "ResNet50": "keras/resnet_50_imagenet",
    "EfficientNetB0": "keras/efficientnet_b0_ra_imagenet",
    "MobileNetV3 Small": "keras/mobilenet_v3_small_100_imagenet"
}


for nombre, repo in repositorios.items():
    print("descargando modelo:", nombre)

    # snapshot_download descarga los archivos del repositorio y devuelve su ruta local
    rutas_modelos[nombre] = snapshot_download(
        repo_id=repo,  # repositorio del modelo en hugging face hub
        token=True     # utiliza las credenciales guardadas de nuestra sesion
    )

    print("modelo descargado en:", rutas_modelos[nombre])

cuenta conectada: saulrodriguezg
descargando modelo: ResNet50


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

modelo descargado en: /root/.cache/huggingface/hub/models--keras--resnet_50_imagenet/snapshots/cfc370bc8a3efa7b2259c7c4e5bde18cdbeff092
descargando modelo: EfficientNetB0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

modelo descargado en: /root/.cache/huggingface/hub/models--keras--efficientnet_b0_ra_imagenet/snapshots/e9658dbecba3601a2ba1bb776e1329688b2c68b2
descargando modelo: MobileNetV3 Small


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

modelo descargado en: /root/.cache/huggingface/hub/models--keras--mobilenet_v3_small_100_imagenet/snapshots/0c0c8625121a1725a05a0ec7d23f0ed22286b3a2


**Carga de modelos**

In [5]:
# usamos tensorflow como backend de keras
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
import keras_hub

# guardamos las rutas de los modelos que descargamos de hugging face
rutas = {
    "ResNet50": rutas_modelos["ResNet50"],
    "EfficientNetB0": rutas_modelos["EfficientNetB0"],
    "MobileNetV3 Small": rutas_modelos["MobileNetV3 Small"]
}

# guardamos los modelos que logremos cargar
modelos = {}

# intentamos cargar cada modelo desde su carpeta local
for nombre, ruta in rutas.items():
    print("cargando modelo:", nombre)
    # mostramos los archivos para comprobar que la carpeta existe
    print(os.listdir(ruta))
    # cargamos el modelo usando el preset correspondiente
    if nombre == "ResNet50":
        preset = "hf://keras/resnet_50_imagenet"
    elif nombre == "EfficientNetB0":
        preset = "hf://keras/efficientnet_b0_ra_imagenet"
    else:
        preset = "hf://keras/mobilenet_v3_small_100_imagenet"
    # cargamos el modelo preentrenado
    modelos[nombre] = keras_hub.models.ImageClassifier.from_preset(preset)

    print("modelo cargado correctamente")

cargando modelo: ResNet50
['task.json', 'config.json', '.gitattributes', 'README.md', 'model.weights.h5', 'metadata.json', 'image_converter.json', 'preprocessor.json', 'task.weights.h5']
modelo cargado correctamente
cargando modelo: EfficientNetB0
['task.json', 'config.json', '.gitattributes', 'README.md', 'model.weights.h5', 'metadata.json', 'image_converter.json', 'preprocessor.json', 'task.weights.h5']
modelo cargado correctamente
cargando modelo: MobileNetV3 Small
['task.json', 'config.json', '.gitattributes', 'README.md', 'model.weights.h5', 'metadata.json', 'image_converter.json', 'preprocessor.json', 'task.weights.h5']
modelo cargado correctamente


**Configuracion de los modelos**

In [6]:
# guardamos los clasificadores que vamos a entrenar
clasificadores = {}

# recorremos los tres modelos preentrenados
for nombre, modelo in modelos.items():
    print("preparando:", nombre)
    # transferencia de aprendizaje: aprovechamos las caracteristicas que el modelo ya aprendio
    base = modelo.backbone
    # congelamos la red para conservar lo aprendido y entrenar solo la nueva capa
    base.trainable = False
    # definimos la entrada para imagenes de 224 x 224 con tres canales de color
    entrada = keras.Input(shape=(224, 224, 3))
    # aplicamos el preprocesamiento que necesita cada modelo
    x = modelo.preprocessor(entrada)
    # extraemos las caracteristicas usando la red preentrenada
    x = base(x, training=False)
    # resumimos las caracteristicas de la imagen
    x = keras.layers.GlobalAveragePooling2D()(x)
    # agregamos una capa nueva para clasificar hojas sanas y enfermas
    salida = keras.layers.Dense(2, activation="softmax")(x)
    # creamos el modelo final para nuestro problema
    clasificador = keras.Model(entrada, salida)
    # configuramos el entrenamiento de la nueva capa
    clasificador.compile(
        optimizer="adam", # metodo para ajustar los pesos
        loss="sparse_categorical_crossentropy", # funcion para clasificacion con etiquetas numericas
        metrics=["accuracy"] # medimos el accuracy
    )
    # guardamos el clasificador para entrenarlo despues
    clasificadores[nombre] = clasificador

print("clasificadores preparados")

preparando: ResNet50
preparando: EfficientNetB0
preparando: MobileNetV3 Small
clasificadores preparados


**Entrenamiento de los modelos**

In [7]:
# guardamos los resultados del entrenamiento
historiales = {}
# entrenamos cada clasificador con las mismas imagenes
for nombre, clasificador in clasificadores.items():
    print("\nentrenando modelo:", nombre)
    # medimos el tiempo que tarda el entrenamiento
    import time
    inicio = time.time()
    # entrenamos con las imagenes y validamos en cada epoca
    historial = clasificador.fit(
        train_ds, # imagenes para entrenar
        validation_data=val_ds, # imagenes para validar
        epochs=5 # numero de vueltas sobre los datos de entrenamiento
    )
    # guardamos el historial para comparar los modelos
    historiales[nombre] = historial
    # mostramos cuanto tardo el entrenamiento
    tiempo = time.time() - inicio
    print("tiempo de entrenamiento:", round(tiempo, 2), "segundos")


entrenando modelo: ResNet50
Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 86s 6s/step - accuracy: 0.6455 - loss: 0.6332 - val_accuracy: 0.7234 - val_loss: 0.5522
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 72s 5s/step - accuracy: 0.6614 - loss: 0.5815 - val_accuracy: 0.7447 - val_loss: 0.5215
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 90s 6s/step - accuracy: 0.6720 - loss: 0.5494 - val_accuracy: 0.7447 - val_loss: 0.4919
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 82s 6s/step - accuracy: 0.7566 - loss: 0.5165 - val_accuracy: 0.7872 - val_loss: 0.4827
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 74s 5s/step - accuracy: 0.8042 - loss: 0.4904 - val_accuracy: 0.8298 - val_loss: 0.4529
tiempo de entrenamiento: 419.55 segundos

entrenando modelo: EfficientNetB0
Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 54s 4s/step - accuracy: 0.6508 - loss: 0.6151 - val_accuracy: 0.7660 - val_loss: 0.5359
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 39s 3s/step - accuracy: 0.7354 - loss: 0.5130 - val_accuracy: 0.8298 - val_loss: 0.4732
Epoch 3/5
12/12 ━━━━

**Guardar los modelos base**

In [8]:
import os

os.makedirs("modelos", exist_ok=True)

for nombre, clasificador in clasificadores.items():
    clasificador.save(f"modelos/{nombre}.keras")
    print("Modelo guardado:", nombre)

Modelo guardado: ResNet50
Modelo guardado: EfficientNetB0
Modelo guardado: MobileNetV3 Small


**Validación de los modelos**

In [10]:
# importamos las metricas que vamos a utilizar
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import time
import pandas as pd

# guardamos los resultados de cada modelo
resultados = []

# evaluamos los tres clasificadores
for nombre, modelo in clasificadores.items():

    # guardamos las etiquetas reales y las predicciones
    etiquetas_reales = []
    predicciones = []

    for imagenes, etiquetas in val_ds:

        # guardamos las etiquetas reales
        etiquetas_reales.extend(etiquetas.numpy())

        # medimos el tiempo de prediccion
        inicio = time.perf_counter()
        probabilidades = modelo.predict(imagenes, verbose=0)

        # elegimos la clase con mayor probabilidad
        predicciones.extend(np.argmax(probabilidades, axis=1))

    # calculamos las metricas de clasificacion
    accuracy = accuracy_score(etiquetas_reales, predicciones)
    precision = precision_score(etiquetas_reales, predicciones, zero_division=0)
    recall = recall_score(etiquetas_reales, predicciones, zero_division=0)
    f1 = f1_score(etiquetas_reales, predicciones, zero_division=0)
    auc = roc_auc_score(etiquetas_reales, predicciones)

    # guardamos los resultados del modelo
    resultados.append({
        "Modelo": nombre,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-score": round(f1, 4),
        "ROC-AUC": round(auc, 4)
    })

# mostramos la tabla comparativa
tabla_resultados = pd.DataFrame(resultados)
display(tabla_resultados)


,Modelo,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,ResNet50,0.8298,0.8250,0.9706,0.8919,0.7161
1,EfficientNetB0,0.8298,0.8611,0.9118,0.8857,0.7636
2,MobileNetV3 Small,0.8085,0.8378,0.9118,0.8732,0.7251


**MLFlow**

In [ ]:
#Instalacion de mlflow
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Configuracion de mlflow**

In [ ]:
import mlflow
#Preparacion de mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Actividad5_hojas_limon")

2026/10/18 06:14:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/18 06:14:28 INFO mlflow.store.db.utils: Updating database tables
2026/10/18 06:14:31 INFO mlflow.tracking.fluent: Experiment with name 'Actividad5_hojas_limon' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1792304071854, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792304071854, lifecycle_stage='active', name='Actividad5_hojas_limon', tags={}, trace_location=None, workspace='default'>

**Registrar el experimento MLflow**

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Actividad5_hojas_limon")
#Registrar los experimentos con las metricas de cada uno
for resultado in resultados:
    nombre = resultado["Modelo"]

    with mlflow.start_run(run_name=nombre):
        mlflow.log_param("modelo", nombre)
        mlflow.log_metric("accuracy", resultado["Accuracy"])
        mlflow.log_metric("precision", resultado["Precision"])
        mlflow.log_metric("recall", resultado["Recall"])
        mlflow.log_metric("f1_score", resultado["F1-score"])

In [ ]:
#Asociacion de los modelos y sus respectivas metricas
rutas_modelos = {
    "EfficientNetB0": "modelos/EfficientNetB0.keras",
    "MobileNetV3 Small": "modelos/MobileNetV3 Small.keras",
    "ResNet50": "modelos/ResNet50.keras"
}

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Actividad5_hojas_limon")

for nombre, ruta in rutas_modelos.items():
    if os.path.exists(ruta):
        with mlflow.start_run(run_name=f"{nombre}_artefacto"):
            mlflow.log_param("modelo", nombre)
            mlflow.log_artifact(ruta, artifact_path="modelo")
        print(f"Artefacto registrado: {nombre}")
    else:
        print(f"No se encontró el archivo: {ruta}")

Artefacto registrado: EfficientNetB0
Artefacto registrado: MobileNetV3 Small
Artefacto registrado: ResNet50


**Verificacion de MLFlow**

In [ ]:
#Ejecucion en segundo plano de MLFlow
!nohup mlflow ui \
  --backend-store-uri sqlite:///mlflow.db \
  --host 0.0.0.0 \
  --port 5000 \
  --allowed-hosts "*" \
  --cors-allowed-origins "*" \
  > mlflow_ui.log 2>&1 &

In [ ]:
!ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=60 -R 80:localhost:5000 nokey@localhost.run


Welcome to localhost.run!

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/



** your connection id is 136.69.251.4:50964, please mention it if you send me a message about an issue. **

6dffbc7438ac03.lhr.life tunneled with tls termination, https://6dffbc7438ac03.lhr.life
create an account and add your key for a longer lasting domain name. see https://localhost.run/docs/forever-free/ for more information.
Open your tunnel address on your mobile with this QR:

                                                      
                                                      
   

**Haciendo el experimento de Grid-Search**

In [ ]:
import numpy as np
import pandas as pd

# conectamos con MLflow y creamos el experimento
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Actividad5_hojas_limon_GridSearch")

# guardamos los nombres de los modelos y sus presets
presets = {
    "ResNet50": "hf://keras/resnet_50_imagenet",
    "EfficientNetB0": "hf://keras/efficientnet_b0_ra_imagenet",
    "MobileNetV3 Small": "hf://keras/mobilenet_v3_small_100_imagenet"
}

# probamos dos tasas de aprendizaje en cada modelo
learning_rates = [0.0001, 0.00001]

# aqui iremos guardando los resultados de cada prueba
resultados_grid = []

# identificamos que numero corresponde a las hojas enfermas
clase_enferma = train_ds.class_names.index("diseased")

# recorremos los tres modelos y probamos cada learning rate
for nombre, preset in presets.items():
    for lr in learning_rates:
        print(f"\nModelo: {nombre} | learning rate: {lr}")

        # cargamos el modelo base desde el preset
        modelo_base = keras_hub.models.ImageClassifier.from_preset(preset)

        # congelamos la red para usar lo que ya aprendio
        base = modelo_base.backbone
        base.trainable = False

        # preparamos la entrada y pasamos la imagen por la red
        entrada = keras.Input(shape=(224, 224, 3))
        x = modelo_base.preprocessor(entrada)
        x = base(x, training=False)

        # agregamos la capa que clasifica las hojas
        x = keras.layers.GlobalAveragePooling2D()(x)
        salida = keras.layers.Dense(2, activation="softmax")(x)

        # armamos el clasificador para este experimento
        clasificador = keras.Model(entrada, salida)

        # configuramos el entrenamiento con el learning rate elegido
        clasificador.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        # guardamos en MLflow los datos de esta prueba
        with mlflow.start_run(run_name=f"{nombre}_lr_{lr}"):
            mlflow.log_param("modelo", nombre)
            mlflow.log_param("learning_rate", lr)
            mlflow.log_param("epochs", 5)

            # entrenamos con las imagenes que ya teniamos separadas
            clasificador.fit(
                train_ds,
                validation_data=val_ds,
                epochs=5,
                verbose=1
            )

            # obtenemos las etiquetas reales de las imagenes de validacion
            y_true = np.concatenate([y.numpy() for _, y in val_ds])

            # el modelo predice si cada hoja esta sana o enferma
            probabilidades = clasificador.predict(val_ds, verbose=0)
            y_pred = np.argmax(probabilidades, axis=1)

            # calculamos las metricas para revisar como le fue
            accuracy = accuracy_score(y_true, y_pred)
            precision = precision_score(
                y_true, y_pred, pos_label=clase_enferma, zero_division=0
            )
            recall = recall_score(
                y_true, y_pred, pos_label=clase_enferma, zero_division=0
            )
            f1 = f1_score(
                y_true, y_pred, pos_label=clase_enferma, zero_division=0
            )

            # registramos las metricas en MLflow
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)

            # agregamos los resultados a la lista para compararlos al final
            resultados_grid.append({
                "Modelo": nombre,
                "Learning rate": lr,
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "F1-score": f1
            })

# mostramos primero los modelos con mejor F1
tabla_grid = pd.DataFrame(resultados_grid)
tabla_grid.sort_values("F1-score", ascending=False)

2026/10/18 07:08:54 INFO mlflow.tracking.fluent: Experiment with name 'Actividad5_hojas_limon_GridSearch' does not exist. Creating a new experiment.



Modelo: ResNet50 | learning rate: 0.0001
Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 76s 6s/step - accuracy: 0.6508 - loss: 0.6640 - val_accuracy: 0.7234 - val_loss: 0.6378
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 61s 5s/step - accuracy: 0.6614 - loss: 0.6518 - val_accuracy: 0.7234 - val_loss: 0.6244
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 77s 6s/step - accuracy: 0.6614 - loss: 0.6432 - val_accuracy: 0.7234 - val_loss: 0.6137
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 67s 6s/step - accuracy: 0.6614 - loss: 0.6352 - val_accuracy: 0.7234 - val_loss: 0.6069
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 68s 6s/step - accuracy: 0.6614 - loss: 0.6291 - val_accuracy: 0.7234 - val_loss: 0.5997

Modelo: ResNet50 | learning rate: 1e-05
Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 76s 6s/step - accuracy: 0.6772 - loss: 0.6554 - val_accuracy: 0.7447 - val_loss: 0.6425
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 69s 6s/step - accuracy: 0.6720 - loss: 0.6538 - val_accuracy: 0.7447 - val_loss: 0.6408
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 81s 6

,Modelo,Learning rate,Accuracy,Precision,Recall,F1-score
4,MobileNetV3 Small,0.00010,0.744681,0.571429,0.307692,0.400000
5,MobileNetV3 Small,0.00001,0.340426,0.263158,0.769231,0.392157
1,ResNet50,0.00001,0.659574,0.000000,0.000000,0.000000
0,ResNet50,0.00010,0.723404,0.000000,0.000000,0.000000
3,EfficientNetB0,0.00001,0.659574,0.000000,0.000000,0.000000
2,EfficientNetB0,0.00010,0.702128,0.000000,0.000000,0.000000


**Guardando los artefactos**

In [ ]:
#Dando el path de los modelos originales
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Actividad5_hojas_limon_GridSearch")

rutas_modelos = {
    "EfficientNetB0": "modelos/EfficientNetB0.keras",
    "MobileNetV3 Small": "modelos/MobileNetV3 Small.keras",
    "ResNet50": "modelos/ResNet50.keras"
}

for nombre, ruta in rutas_modelos.items():
    if os.path.exists(ruta):
        with mlflow.start_run(run_name=f"{nombre}_modelo_original_artefacto"):
            mlflow.log_param("modelo", nombre)
            mlflow.log_param(
                "tipo",
                "modelo original previamente entrenado; no corresponde al Grid Search"
            )
            mlflow.log_artifact(ruta, artifact_path="modelo")

        print(f"Registrado: {nombre}")
    else:
        print(f"No encontré el archivo: {ruta}")

Registrado: EfficientNetB0
Registrado: MobileNetV3 Small
Registrado: ResNet50
